In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")
spark.conf.set("spark.microsoft.delta.stats.collect.fromArrow", "false")

StatementMeta(, cf0a89e1-334b-4ea1-bd49-5a85e6ba5d51, 3, Finished, Available, Finished, False)

In [2]:
silver_df = spark.read.table("silver_weather_data")
silver_df.printSchema()

StatementMeta(, cf0a89e1-334b-4ea1-bd49-5a85e6ba5d51, 4, Finished, Available, Finished, False)

root
 |-- weather_key: long (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- country: string (nullable = true)
 |-- humidity: long (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- city: string (nullable = true)
 |-- localtime: string (nullable = true)
 |-- tz_id: string (nullable = true)
 |-- pressure_mb: double (nullable = true)
 |-- region: string (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- windkph: double (nullable = true)
 |-- comfortindex: double (nullable = true)
 |-- date: date (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)



In [3]:
from pyspark.sql import functions as F
daily_summary = "gold_daily_summary"

df_daily_summary = silver_df \
    .groupBy("date", "month", "year", "is_weekend", "temperature_c") \
    .agg (
        F.round(F.avg("temperature_c")).alias("avg_temp"),
        F.round(F.min("temperature_c")).alias("min_temp"),
        F.round(F.max("temperature_c")).alias("max_temp"),
        F.round(F.avg("humidity")).alias("avg_humidity"),
        F.round(F.avg("windkph")).alias("avg_windkph"),
        F.round(F.avg("comfortindex")).alias("avg_comfortindex"),
        F.first("weather_condition").alias("weather_condition")

    ) \
    .withColumn("is_comfortable", F.when(F.col("avg_comfortindex") < 50, True).otherwise(False))

df_daily_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(daily_summary)


display(df_daily_summary)


StatementMeta(, cf0a89e1-334b-4ea1-bd49-5a85e6ba5d51, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e140ea54-e320-42f1-83e7-181f15f06cf2)

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
weekly_summary = "gold_weekly_summary"
week_window = Window.orderBy("year", "week_number")

gold_daily_summary = spark.read.table("gold_daily_summary")

df_weekly_summary = gold_daily_summary \
    .groupBy("date", "month", "year", F.weekofyear("date").alias("week_number"), "avg_temp", "avg_comfortindex") \
    .agg (
        F.sum(F.when(F.col("is_comfortable"), 1).otherwise(0)).alias("comfort_count")

    ) \
    .withColumn("prev_week_temp", F.lag("avg_temp", 1).over(week_window)) \
    .withColumn("week_over_week_change", F.col("avg_temp") - F.col("prev_week_temp"))

StatementMeta(, cf0a89e1-334b-4ea1-bd49-5a85e6ba5d51, 6, Finished, Available, Finished, False)

In [5]:
df_weekly_summary.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(weekly_summary)


StatementMeta(, cf0a89e1-334b-4ea1-bd49-5a85e6ba5d51, 7, Finished, Available, Finished, False)

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

monthly_summary = "gold_monthly_summary"

month_window = Window.orderBy("year", "month")

df_monthly_summary = silver_df \
    .groupBy("date", "year", "month") \
    .agg (
        F.round(F.avg("temperature_c")).alias("avg_temp"),
        F.round(F.min("temperature_c")).alias("min_temp"),
        F.round(F.max("temperature_c")).alias("max_temp"),
        F.mode(F.col("weather_condition")).alias("most_common_condition")


    ) \
    .withColumn("month_year", F.date_format("date", "MMM-yyyy")) \
    .withColumn("prev_month_temp", F.lag("avg_temp", 1).over(month_window)) \
    .withColumn("month_over_month", F.col("avg_temp") - F.col("prev_month_temp"))




df_monthly_summary.write \
.mode("overwrite") \
.format("delta") \
.option("overwriteSchema", "true") \
.saveAsTable(monthly_summary)

# display(df_monthly_summary)

StatementMeta(, cf0a89e1-334b-4ea1-bd49-5a85e6ba5d51, 9, Finished, Available, Finished, False)